# Is a Step-Free Station Enough?

This notebook audits TfL's detailed platform and station topology data, a dated live lift-disruption snapshot, and the existing typical-day Tube flow model. It asks what station records can tell us about access, and where they stop short of describing a complete journey.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px


ROOT = next(
    parent
    for parent in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (parent / "data/tube_crowding/stations_master.csv").exists()
    and (parent / "london_tube_crowding/step_free_access.py").exists()
)
PROJECT_DIR = ROOT / "london_tube_crowding"
DATA_DIR = ROOT / "data" / "tube_crowding"
EXPORT_DIR = PROJECT_DIR / "exports"
sys.path.insert(0, str(PROJECT_DIR))

import step_free_access as access


LIFT_SNAPSHOT_DATE = "2026-10-09"
stations = pd.read_csv(DATA_DIR / "stations_master.csv", dtype=str).fillna("")
crowding = pd.read_csv(DATA_DIR / "station_line_crowding.csv")
topology = access.load_topology_tables(DATA_DIR / "tfl-stationdata-detailed.zip")
platform_services = access.build_platform_service_table(
    topology["PlatformServices"], stations
)
lift_disruptions = access.load_lift_disruptions(
    DATA_DIR / f"lift-disruptions-{LIFT_SNAPSHOT_DATE}.json", stations
)

print(f"Tube station records: {len(stations)} ({stations['name'].nunique()} names)")
print(f"Detailed topology tables: {len(topology)}")
print(
    "Platform service records matched to Tube stations: "
    f"{len(platform_services):,} across "
    f"{platform_services['station_naptan'].nunique()} stations"
)
print(
    "Lift disruption records matched to Tube stations: "
    f"{lift_disruptions['matched_tube_station'].sum()} of {len(lift_disruptions)}"
)


Tube station records: 272 (270 names)
Detailed topology tables: 11
Platform service records matched to Tube stations: 851 across 272 stations
Lift disruption records matched to Tube stations: 14 of 22


## Platform Service Records

The topology is platform- and line-specific. The flags below are reported as TfL supplied them; blanks are counted as missing, not as a negative access condition. Step and gap fields are summarized for coverage only until their units and interpretation are confirmed from TfL's guide.

In [2]:
service_audit = platform_services.copy()
service_audit["manual_ramp_flag"] = (
    service_audit["level_access_by_manual_ramp"]
    .str.strip()
    .str.upper()
    .replace({"": "UNKNOWN"})
)
service_audit["designated_access_flag"] = (
    service_audit["designated_level_access_point"]
    .str.strip()
    .str.upper()
    .replace({"": "UNKNOWN"})
)
step_gap_columns = ["min_step", "max_step", "average_step", "min_gap", "max_gap", "average_gap"]
service_audit["has_any_step_gap_measure"] = service_audit[step_gap_columns].ne("").any(axis=1)

platform_coverage = (
    service_audit.groupby("line", dropna=False)
    .agg(
        service_records=("platform_unique_id", "size"),
        stations=("station_naptan", "nunique"),
        manual_ramp_true=("manual_ramp_flag", lambda values: values.eq("TRUE").sum()),
        manual_ramp_false=("manual_ramp_flag", lambda values: values.eq("FALSE").sum()),
        designated_point_true=("designated_access_flag", lambda values: values.eq("TRUE").sum()),
        step_gap_records=("has_any_step_gap_measure", "sum"),
    )
    .sort_values("service_records", ascending=False)
)

display(platform_coverage)
print("Manual-ramp flag values:", service_audit["manual_ramp_flag"].value_counts().to_dict())
print("Designated-point flag values:", service_audit["designated_access_flag"].value_counts().to_dict())
print(
    "Records with at least one step/gap value:",
    int(service_audit["has_any_step_gap_measure"].sum()),
    "of",
    len(service_audit),
)


,service_records,stations,manual_ramp_true,manual_ramp_false,designated_point_true,step_gap_records
line,,,,,,
district,134,60,23,111,19,62
northern,120,52,9,111,37,51
piccadilly,117,53,33,84,19,44
central,107,49,33,74,0,17
metropolitan,84,35,20,64,23,39
circle,80,36,12,68,10,28
hammersmith-city,63,29,5,58,14,31
jubilee,57,27,8,49,11,40
bakerloo,52,25,11,41,0,5


Manual-ramp flag values: {'FALSE': 694, 'TRUE': 157}
Designated-point flag values: {'FALSE': 698, 'TRUE': 153}
Records with at least one step/gap value: 339 of 851


## Step-Free Interchange Links

TfL publishes explicit platform-to-platform interchange distances in a separate table. This is a link between two platform records, not a statement that every route through a station is accessible. The join below only reports connections where both platform IDs resolve to the Tube service table.

In [3]:
platform_lookup = platform_services[
    ["platform_unique_id", "station_naptan", "station_name", "line"]
].drop_duplicates("platform_unique_id")
interchanges = topology["StepFreeIntechangeInfo"].copy()
interchanges = interchanges.merge(
    platform_lookup.add_prefix("from_"),
    how="left",
    left_on="FromPlatformUniqueId",
    right_on="from_platform_unique_id",
)
interchanges = interchanges.merge(
    platform_lookup.add_prefix("to_"),
    how="left",
    left_on="ToPlatformUniqueId",
    right_on="to_platform_unique_id",
)
from_matched = interchanges["from_station_naptan"].notna() & interchanges[
    "from_station_naptan"
].ne("")
to_matched = interchanges["to_station_naptan"].notna() & interchanges[
    "to_station_naptan"
].ne("")
matched_interchanges = interchanges[from_matched & to_matched].copy()

print(f"All published step-free interchange links: {len(interchanges)}")
print(f"Links with both endpoints matched to Tube service records: {len(matched_interchanges)}")
print(f"Distinct stations represented: {matched_interchanges['from_station_naptan'].nunique()}")
display(
    matched_interchanges[
        ["from_station_name", "from_line", "to_station_name", "to_line", "DistanceInMetres"]
    ].sort_values(["from_station_name", "from_line", "to_line"])
)


All published step-free interchange links: 114
Links with both endpoints matched to Tube service records: 86
Distinct stations represented: 12


,from_station_name,from_line,to_station_name,to_line,DistanceInMetres
18,Epping,central,Epping,central,450
19,Epping,central,Epping,central,450
36,Green Park,jubilee,Green Park,piccadilly,220
37,Green Park,jubilee,Green Park,piccadilly,220
40,Green Park,jubilee,Green Park,piccadilly,220
...,...,...,...,...,...
53,Woodford,central,Woodford,central,130
54,Woodford,central,Woodford,central,130
55,Woodford,central,Woodford,central,130
56,Woodside Park,northern,Woodside Park,northern,600


## Lift Disruptions on 9 October 2026

The live feed is a snapshot, not a history. It contains station-level notices with free-text messages; several stations have more than one record, and some notices describe a partial outage or an alternative entrance. Count records and affected stations separately, then read each message before describing the impact.

In [4]:
matched_lifts = lift_disruptions[lift_disruptions["matched_tube_station"]].copy()
lift_station_summary = (
    matched_lifts.groupby(["station_naptan", "station_name"], as_index=False)
    .agg(
        notice_records=("message", "size"),
        lift_ids=(
            "lift_ids",
            lambda values: "; ".join(
                sorted({lift_id for value in values for lift_id in value.split(";") if lift_id})
            ),
        ),
        messages=("message", lambda values: "\n\n".join(values)),
    )
    .sort_values(["notice_records", "station_name"], ascending=[False, True])
)

print(f"Feed records: {len(lift_disruptions)}")
print(f"Records matched to Tube stations: {len(matched_lifts)}")
print(f"Distinct Tube stations with one or more notices: {len(lift_station_summary)}")
display(lift_station_summary[["station_name", "notice_records", "lift_ids"]])

selected_names = ["Canada Water", "Earl's Court", "Wembley Park"]
display(
    matched_lifts[matched_lifts["station_name"].isin(selected_names)][
        ["station_name", "lift_ids", "message"]
    ]
)


Feed records: 22
Records matched to Tube stations: 14
Distinct Tube stations with one or more notices: 10


,station_name,notice_records,lift_ids
4,Earl's Court,3,940GZZLUECT-Lift-5
3,Elephant & Castle,2,HUBEPH-Lift-5; HUBEPH-Lift-6
9,Wembley Park,2,940GZZLUWYP-Lift-5
0,Bank,1,HUBBAN-Lift-7
1,Canada Water,1,HUBZCW-Lift-4
2,Canary Wharf,1,HUBCAW-Lift-LU 2
5,Harrow & Wealdstone,1,HUBHRW-Lift-1
6,North Greenwich,1,940GZZLUNGW-Lift-1
8,Wembley Central,1,HUBWMB-Lift-3
7,Willesden Junction,1,HUBWIJ-Lift-1


,station_name,lift_ids,message
0,Wembley Park,940GZZLUWYP-Lift-5,WEMBLEY PARK STATION: From Monday 10 March unt...
8,Earl's Court,940GZZLUECT-Lift-5,Earls Court: No Step Free Access - Step free a...
10,Earl's Court,940GZZLUECT-Lift-5,Earl's Court: No Step Free Access - Step free ...
16,Wembley Park,940GZZLUWYP-Lift-5,Wembley Park: No Step Free Access - Step free ...
19,Canada Water,HUBZCW-Lift-4,"CANADA WATER STATION: Until Spring 2027, no st..."
20,Earl's Court,940GZZLUECT-Lift-5,EARLS COURT STATION: From Monday 28 September ...


## What the Crowding Data Adds

The existing passenger-flow feed describes a modelled typical day, summed across line/time components in the crowding project. This comparison adds network context to the affected stations; it does not measure platform density, lift queues, accessible-route congestion, or anyone's actual journey.

In [5]:
crowding["value"] = pd.to_numeric(crowding["value"], errors="raise")
typical_station_flow = (
    crowding.groupby("station_naptan", as_index=False)
    .agg(typical_day_flow_units=("value", "sum"))
)
lift_crowding_context = lift_station_summary.merge(
    typical_station_flow,
    how="left",
    on="station_naptan",
    validate="one_to_one",
)
print(
    "Affected Tube stations with crowding data:",
    int(lift_crowding_context["typical_day_flow_units"].notna().sum()),
    "of",
    len(lift_crowding_context),
)
display(
    lift_crowding_context[
        ["station_name", "notice_records", "typical_day_flow_units"]
    ].sort_values("typical_day_flow_units", ascending=False)
)


Affected Tube stations with crowding data: 10 of 10


,station_name,notice_records,typical_day_flow_units
3,Bank,1,184035
5,Canary Wharf,1,93308
0,Earl's Court,3,91060
4,Canada Water,1,75234
2,Wembley Park,2,48905
1,Elephant & Castle,2,41901
7,North Greenwich,1,38881
9,Willesden Junction,1,11373
8,Wembley Central,1,6202
6,Harrow & Wealdstone,1,5171


## Map of the Captured Lift Notices

The map marks only stations with one or more records in the 9 October snapshot. A grey point means no matching record appeared in this particular response; it does not certify that lifts were working or that the station is step-free.

In [6]:
map_data = stations.merge(
    lift_crowding_context[
        ["station_naptan", "notice_records", "messages", "typical_day_flow_units"]
    ],
    how="left",
    left_on="naptan",
    right_on="station_naptan",
    validate="one_to_one",
)
map_data["notice_records"] = map_data["notice_records"].fillna(0).astype(int)
map_data["messages"] = map_data["messages"].fillna("")
map_data["lift_snapshot_status"] = map_data["notice_records"].map(
    lambda count: "Lift notice in snapshot" if count else "No matching notice in snapshot"
)
map_data["marker_size"] = 9

fig = px.scatter_map(
    map_data,
    lat="lat",
    lon="lon",
    color="lift_snapshot_status",
    size="marker_size",
    size_max=9,
    hover_name="name",
    hover_data={
        "lift_snapshot_status": False,
        "notice_records": True,
        "messages": True,
        "typical_day_flow_units": ":,",
        "lat": False,
        "lon": False,
        "marker_size": False,
    },
    color_discrete_map={
        "No matching notice in snapshot": "#8b969b",
        "Lift notice in snapshot": "#c0392b",
    },
    zoom=9.2,
    center={"lat": 51.52, "lon": -0.13},
    map_style="carto-positron",
)
fig.update_layout(
    title="Lift-disruption notices in TfL's 9 October 2026 snapshot",
    font={"family": "Inter, system-ui, sans-serif", "size": 13},
    template="plotly_white",
    margin={"l": 0, "r": 0, "t": 55, "b": 0},
    legend_title_text="TfL feed record",
)
fig.update_traces(marker={"opacity": 0.85})

EXPORT_DIR.mkdir(exist_ok=True)
map_export = EXPORT_DIR / "step-free-stations.html"
fig.write_html(
    map_export,
    include_plotlyjs="cdn",
    full_html=True,
    config={"responsive": True, "displaylogo": False},
    auto_play=False,
)
html = map_export.read_text(encoding="utf-8")
if 'name="viewport"' not in html:
    html = html.replace(
        '<meta charset="utf-8" />',
        '<meta charset="utf-8" /><meta name="viewport" content="width=device-width, initial-scale=1" />',
        1,
    )
    map_export.write_text(html, encoding="utf-8")
print(f"Wrote {map_export}")


Wrote /Users/Akhilesh.Koul/Documents/GitHub/CodePlayground/london_tube_crowding/exports/step-free-stations.html


## Interchange Record Duplication Check

The source table may list repeated or reciprocal platform links. Keep its raw row count separate from the number of unique directed and undirected platform pairs before quoting a total.

In [7]:
directed_pairs = matched_interchanges[
    ["FromPlatformUniqueId", "ToPlatformUniqueId", "DistanceInMetres"]
].drop_duplicates()
undirected_pairs = {
    tuple(sorted((from_platform, to_platform)))
    for from_platform, to_platform in directed_pairs[
        ["FromPlatformUniqueId", "ToPlatformUniqueId"]
    ].itertuples(index=False, name=None)
}
print(f"Matched source rows: {len(matched_interchanges)}")
print(f"Unique directed platform pairs: {len(directed_pairs)}")
print(f"Unique undirected platform pairs: {len(undirected_pairs)}")
print(
    "Rows removed as exact repeated directed records:",
    len(matched_interchanges) - len(directed_pairs),
)


Matched source rows: 86
Unique directed platform pairs: 74
Unique undirected platform pairs: 37
Rows removed as exact repeated directed records: 12


## Sources and Limits

- TfL detailed station topology archive, downloaded 9 October 2026: [stationdata-detailed.zip](https://api.tfl.gov.uk/stationdata/tfl-stationdata-detailed.zip). TfL's [step-free access and toilet data guide](https://tfl.gov.uk/cdn/static/cms/documents/step-free-access-and-toilet-data-guide.pdf) describes the feed. The analysis does not treat a route-information flag as proof of accessible travel.
- TfL [live lift disruption feed](https://api.tfl.gov.uk/Disruptions/Lifts/v2), captured 9 October 2026. Messages are a one-time snapshot, can be duplicated, and can describe partial outages or alternative entrances.
- The typical-day flow file is from the existing [Tube crowding analysis](https://github.com/koulakhilesh/CodePlayground/tree/feature/tfl-step-free-access/london_tube_crowding). It is not live and does not measure the conditions around lifts or accessible paths.

The service join uses `StopAreaNaptanCode` to match all 272 station-master NaPTANs. It yields 851 platform-service rows. A step or gap value appears in at least one of six fields on 339 rows; the other rows remain unknown. TfL lists 114 step-free interchange records; 86 rows resolve to two Tube service endpoints, representing 37 unique undirected platform pairs across 12 stations. The remaining records are left unmatched.

This is a station/platform data audit, not an end-to-end route assessment or travel recommendation. The data contains no account of signage, staff assistance, a passenger's full origin/destination, or a person's experience. Field reporting is still needed for those questions.
